# Job Market Trends - Exploratory Data Analysis

This notebook analyzes the cleaned and enriched job market dataset to identify key trends in salaries, remote work, and **skills demand**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Cleaned Data

In [ ]:
df = pd.read_csv('../linkedin_jobs_dashboard_ready.csv')
df.info()
df.head()

## 1.5 Null Audit

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(null_pct[null_pct > 0].sort_values(ascending=False))

## 2. Salary Analysis

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[df['normalized_salary'].notna()], x='experience_group', y='normalized_salary', order=['Entry', 'Mid-level', 'Senior'])
plt.title('Salary Distribution by Experience Level')
plt.ylabel('Normalized Salary (USD)')
plt.xlabel('Experience Group')
plt.show()

## 2.5 Remote vs Onsite Salary Gap

In [ ]:
sns.boxplot(data=df[df['normalized_salary'].notna()], 
            x='is_remote', y='normalized_salary')
plt.title('Salary: Remote vs Onsite')
plt.show()

## 3. Remote Work Trends

In [ ]:
remote_counts = df['is_remote'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(remote_counts, labels=remote_counts.index, autopct='%1.1f%%', colors=['#66b3ff','#99ff99'])
plt.title('Remote vs. Onsite Job Distribution')
plt.show()

## 3.5 Monthly Job Posting Trend

In [ ]:
monthly = df.groupby(['year', 'month']).size().reset_index(name='job_count')
monthly['date'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))
sns.lineplot(data=monthly, x='date', y='job_count')
plt.title('Job Postings Over Time')
plt.show()

## 4. Top Skills in High Demand

In [ ]:
# Flatten the top_skills column (comma-separated strings)
skills_series = df['top_skills'].str.split(', ').explode().value_counts().head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x=skills_series.values, y=skills_series.index, palette='crest')
plt.title('Top 15 Most In-Demand Skills')
plt.xlabel('Frequency Across Job Postings')
plt.ylabel('Skill Name')
plt.show()

## 4.5 Skills vs Salary

In [ ]:
df_skills = pd.read_csv('../linkedin_skills_exploded.csv')
skill_salary = df_skills.groupby('skill')['normalized_salary'].median().sort_values(ascending=False).head(20)
sns.barplot(x=skill_salary.values, y=skill_salary.index, palette='rocket')
plt.title('Top 20 Skills by Median Salary')
plt.show()

## 5. Top Industries by Job Volume

In [ ]:
top_industries = df['industry_name'].str.split(', ').explode().value_counts().head(10)
sns.barplot(x=top_industries.values, y=top_industries.index, palette='viridis')
plt.title('Top 10 Industries by Number of Job Postings')
plt.xlabel('Count')
plt.show()

## 5.5 Median Salary by Industry

In [ ]:
df_ind = df.assign(industry=df['industry_name'].str.split(', ')).explode('industry')
industry_salary = df_ind.groupby('industry')['normalized_salary'].median().sort_values(ascending=False).head(15)
sns.barplot(x=industry_salary.values, y=industry_salary.index)
plt.title('Median Salary by Industry (Top 15)')
plt.show()

## 6. Demand Score vs. Company Size

In [ ]:
avg_demand = df.groupby('company_size')['demand_score'].mean().sort_values(ascending=False)
sns.barplot(x=avg_demand.index, y=avg_demand.values, palette='magma')
plt.title('Average Demand Score by Company Size')
plt.ylabel('Avg Demand Score')
plt.xlabel('Company Size Category')
plt.show()

## 7. Correlation Heatmap


In [ ]:
corr_cols = ['normalized_salary', 'demand_score', 'employee_count', 'views', 'applies']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlations')
plt.show()